In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp,col

def write_to_gold(input_df,table_name,merge_condition,columns_to_update):
    
    input_df = (input_df.withColumn("created_timestamp",current_timestamp())
                .withColumn("updated_timestamp",current_timestamp())
                  )
    if not spark.catalog.tableExists(table_name):
            input_df.write.mode("overwrite").saveAsTable(table_name)
    else:
            delta_table = DeltaTable.forName(spark,table_name)
            update_map = {column : f"s.{column}" for column in columns_to_update}
            update_map["updated_timestamp"] = "s.updated_timestamp"
            (
                delta_table.alias("t")
                   .merge(input_df.alias("s"),merge_condition)
                   .whenMatchedUpdate(
                       set=update_map)
                   .whenNotMatchedInsertAll()
                   .execute()
            )